In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import load_fine_tuned_model, predict_sentiment_batch


C:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
id_to_label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
models_dir = project_root / "models"
predictions_dir = project_root / "outputs" / "predictions"

predictions_dir.mkdir(parents=True, exist_ok=True)

news_df = pd.read_csv(
    processed_dir / "fnspid_tsla_2019_2023.csv"
)

news_df["Date"] = pd.to_datetime(
    news_df["Date"],
    utc=True
)

news_df = news_df.sort_values("Date").reset_index(drop=True)

print("Headlines to score:", len(news_df))
print("Date range:", news_df["Date"].min(), "to", news_df["Date"].max())

Headlines to score: 872
Date range: 2019-07-01 00:00:00+00:00 to 2023-12-16 22:00:00+00:00


In [3]:
model_path = models_dir / 'distilbert_financial_sentiment'
model, tokenizer, device = load_fine_tuned_model(model_path)
print('Loaded saved fine-tuned DistilBERT model.')


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3015.82it/s]

Loaded saved fine-tuned DistilBERT model.


In [4]:

news_dataset = Dataset.from_pandas(
    news_df[["Article_title"]],
    preserve_index=False
)

def tokenize_function(batch):
    return tokenizer(
        batch["Article_title"],
        truncation=True,
        max_length=128
    )

tokenized_news = news_dataset.map(
    tokenize_function,
    batched=True
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map: 100%|██████████| 872/872 [00:00<00:00, 13281.21 examples/s]

In [5]:

inference_args = TrainingArguments(
    output_dir=str(models_dir / "historical_inference"),
    per_device_eval_batch_size=16,
    report_to="none"
)

inference_trainer = Trainer(
    model=model,
    args=inference_args,
    data_collator=data_collator,
    processing_class=tokenizer
)

prediction_output = inference_trainer.predict(
    tokenized_news
)

logits = prediction_output.predictions

if isinstance(logits, tuple):
    logits = logits[0]

predicted_ids = np.argmax(logits, axis=1)

shifted_logits = logits - np.max(
    logits,
    axis=1,
    keepdims=True
)

probabilities = np.exp(shifted_logits)
probabilities = probabilities / probabilities.sum(
    axis=1,
    keepdims=True
)

print("Completed headline scoring.")

C:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Completed headline scoring.


In [6]:

scored_news = news_df.copy()

scored_news["predicted_label"] = [
    id_to_label[prediction]
    for prediction in predicted_ids
]

scored_news["negative_probability"] = probabilities[:, 0]
scored_news["neutral_probability"] = probabilities[:, 1]
scored_news["positive_probability"] = probabilities[:, 2]

scored_news["confidence"] = probabilities.max(axis=1)

scored_news["sentiment_score"] = (
    scored_news["positive_probability"]
    - scored_news["negative_probability"]
)

display(
    scored_news[
        [
            "Date",
            "Article_title",
            "predicted_label",
            "confidence",
            "sentiment_score"
        ]
    ].tail(10)
)

,Date,Article_title,predicted_label,confidence,sentiment_score
862,2023-12-07 00:00:00+00:00,Tata Motors lobbies India not to lower EV impo...,neutral,0.833882,0.057546
863,2023-12-08 00:00:00+00:00,Vanguard Total World Stock ETF Experiences Big...,positive,0.829839,0.810482
864,2023-12-09 00:00:00+00:00,3 Stocks Transforming the Electric Vehicle Cha...,positive,0.783256,0.769468
865,2023-12-10 00:00:00+00:00,Stellantis Just Passed Ford and GM in the Elec...,neutral,0.822959,0.111362
866,2023-12-11 00:00:00+00:00,3 High-Voltage Battery Stocks to Ride EV Growt...,positive,0.929505,0.914388
867,2023-12-12 00:00:00+00:00,TSLA Factor-Based Stock Analysis,neutral,0.971663,0.012238
868,2023-12-13 00:00:00+00:00,Why Tesla's Cheap (2024 Outlook),neutral,0.917476,0.035566
869,2023-12-14 00:00:00+00:00,Tesla Stock Can Soar in 2024 if It Does These ...,neutral,0.921092,0.059746
870,2023-12-15 00:00:00+00:00,Elon Musk says oil and gas should not be demon...,neutral,0.930277,-0.017325
871,2023-12-16 22:00:00+00:00,Scotiabank's Mexico bet eyeing $1.6 trillion N...,neutral,0.869269,0.038845


In [7]:

output_file = (
    predictions_dir
    / "tsla_scored_headlines_2019_2023.csv"
)

scored_news.to_csv(output_file, index=False)

print("Saved scored headlines to:")
print(output_file)

Saved scored headlines to:
C:\Users\sarth\OneDrive\Desktop\Plasksha Work\Python\time-aware-financial-news-monitor\outputs\predictions\tsla_scored_headlines_2019_2023.csv
